# CIS3120 LibraryDB Assignment

This notebook creates the library database and runs the required queries.

## Housekeeping (Step 1)

In [1]:
import sqlite3
import csv
import urllib.request
import os

BASE_URL = "https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/CIS3120-BMWB/refs/heads/main/"

BOOK_URL = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL = BASE_URL + "Loan.csv"

BOOK_PATH = "Book.csv"
MEMBER_PATH = "Member.csv"
LOAN_PATH = "Loan.csv"

DB_PATH = "library.db"


## Download CSV Files (Step 2)

In [2]:
urllib.request.urlretrieve(BOOK_URL, BOOK_PATH)
urllib.request.urlretrieve(MEMBER_URL, MEMBER_PATH)
urllib.request.urlretrieve(LOAN_URL, LOAN_PATH)

print("CSV files downloaded")


CSV files downloaded


## Create Database and Tables (Step 3)

In [3]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

conn.execute("""
CREATE TABLE IF NOT EXISTS Book (
    callNo TEXT NOT NULL,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    PRIMARY KEY (callNo)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Member (
    id INTEGER NOT NULL,
    firstname TEXT NOT NULL,
    lastName TEXT NOT NULL,
    PRIMARY KEY (id)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Loan (
    callNo TEXT NOT NULL,
    id INTEGER NOT NULL,
    dateBorrowed TEXT NOT NULL,
    dateReturned TEXT,
    dateDue TEXT NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id) REFERENCES Member(id)
);
""")

conn.commit()
print("Tables created")

print(conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall())


Tables created
[('Book',), ('Loan',), ('Member',)]


## Load Book Data (Step 4)

In [4]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?)",
            (row['callNo'], row['title'], row['author'])
        )

conn.commit()
print("Book rows loaded:", conn.execute("SELECT COUNT(*) FROM Book").fetchone()[0])


Book rows loaded: 11


## Load Member Data (Step 5)

In [5]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?)",
            (int(row['id']), row['firstname'], row['lastName'])
        )

conn.commit()
print("Member rows loaded:", conn.execute("SELECT COUNT(*) FROM Member").fetchone()[0])


Member rows loaded: 4


## Load Loan Data (Step 6)

In [6]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['dateReturned'].strip() == "":
            date_returned = None
        else:
            date_returned = row['dateReturned']

        conn.execute(
            """
            INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
            VALUES (?, ?, ?, ?, ?)
            """,
            (row['callNo'], int(row['id']), row['dateBorrowed'], date_returned, row['dateDue'])
        )

conn.commit()
print("Loan rows loaded:", conn.execute("SELECT COUNT(*) FROM Loan").fetchone()[0])


Loan rows loaded: 4


## Query 1 — All Books

Retrieve all columns from the Book table, ordered alphabetically by author.

In [7]:
query = """
SELECT *
FROM Book
ORDER BY author;
"""

for row in conn.execute(query):
    print(row)


('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


## Query 2 — Books Not Yet Returned

Show the title and member name for books where dateReturned is NULL.

In [8]:
query = """
SELECT Book.title, Member.firstname, Member.lastName
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
JOIN Member ON Loan.id = Member.id
WHERE Loan.dateReturned IS NULL;
"""

for row in conn.execute(query):
    print(row)


("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


## Query 3 — Loan History for a Specific Book

Show the loan history for call number R 141 E45 2006.

In [9]:
query = """
SELECT Member.firstname || ' ' || Member.lastName AS fullName, 
        Loan.dateBorrowed, 
        Loan.dateDue, 
        Loan.dateReturned
FROM Loan
JOIN Member ON Loan.id = Member.id
WHERE Loan.callNo = 'R 141 E45 2006'
ORDER BY Loan.dateBorrowed;
"""

for row in conn.execute(query):
    print(row)


('Betty Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


## Query 4 — Members Who Have Never Borrowed a Book

Find members who do not have any row in the Loan table.

In [10]:
query = """
SELECT Member.id, Member.firstname, Member.lastName
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
WHERE Loan.id IS NULL;
"""

for row in conn.execute(query):
    print(row)


(4, 'John', 'Martin')


## Query 5 — Count of Loans per Member

Count how many loans each member has, including members with zero loans.

In [11]:
query = """
SELECT Member.firstname || ' ' || Member.lastName AS fullName,
       COUNT(Loan.callNo) AS totalLoans
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
GROUP BY Member.id, Member.firstname, Member.lastName
ORDER BY totalLoans DESC;
"""

for row in conn.execute(query):
    print(row)


('David Martin', 2)
('John Smith', 1)
('Betty Freeman', 1)
('John Martin', 0)


## Query 6 — Free-Choice Analytical Query

Business question: Which authors have more than one book in the library? This is useful because the library can see which authors have a larger presence in the collection.

In [12]:
query = """
SELECT author, COUNT(*) AS book_count
FROM Book
GROUP BY author
HAVING COUNT(*) > 1
ORDER BY book_count DESC;
"""

for row in conn.execute(query):
    print(row)


('Joe Celko', 3)


## Summary

One data quality issue I noticed is that some loan records have a blank dateReturned value. I converted those blank values to None so SQLite stores them as NULL. This matters because the queries use IS NULL to find books that have not been returned yet. A limitation of this dataset is that it is very small, with only a few members and loans. In a real library system, there would probably be more tables for things like copies, branches, fines, and member contact information.

In [13]:
conn.close()
print("Database connection closed")


Database connection closed
